In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.types import FloatType
from pyspark.sql.functions import col, upper, when, cast
import time

spark = SparkSession.builder \
    .appName("Processador_Gigante_5GB") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .getOrCreate() 
    # maxPartitionBytes = 128MB (O Spark vai quebrar o arquivo de 5GB em pedacinhos de 128MB)

print(spark.version)
ARQUIVO_CSV = "work/dados/particao_unica" 

3.5.0


In [3]:
start_time = time.time()

df = spark.read.csv(
    ARQUIVO_CSV, 
    header=True, 
    inferSchema=False, # facilita leitura
    sep=","
)

print(f"Tempo para mapear o arquivo (Lazy): {time.time() - start_time:.2f} segundos")
print(f"Número estimado de partições: {df.rdd.getNumPartitions()}")


Tempo para mapear o arquivo (Lazy): 1.76 segundos
Número estimado de partições: 44


In [7]:
df.show(5)

+---+------------------+---------+--------------------+
| id|             valor|   status|      transacao_hash|
+---+------------------+---------+--------------------+
|  0| 555.8546829144523| Aprovado|e7b5eb1a-8014-45f...|
|  1| 940.1863788688013|Reprovado|280c94fe-a7b0-4e9...|
|  2| 265.4064366417952|Reprovado|d7c73c22-5a8c-4fc...|
|  3|   97.720036292681|Reprovado|5b4805de-3d71-413...|
|  4|11.703993495950193| Aprovado|6fce569e-427c-445...|
+---+------------------+---------+--------------------+
only showing top 5 rows



In [8]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- status: string (nullable = true)
 |-- transacao_hash: string (nullable = true)



In [9]:
coluna_filtro = input("\nDigita o nome da coluna para FILTRAR: ")
valor_filtro = input(f"Digita o valor que você quer buscar na coluna '{coluna_filtro}': ")

print(f"Configurando filtro: {coluna_filtro} == '{valor_filtro}'")
df_filtrado = df.filter(col(coluna_filtro) == valor_filtro)


Digita o nome da coluna para FILTRAR:  status
Digita o valor que você quer buscar na coluna 'status':  Aprovado


Configurando filtro: status == 'Aprovado'


In [11]:
df_final = df_filtrado.withColumn(
    "valor_float",
    col("valor").cast(FloatType())
)

In [12]:
print("iniciando processamento...")
start_proc = time.time()

total_linhas = df_final.count()

tempo_total = time.time() - start_proc
print("concluido")
print(f"total linhas: {total_linhas}")
print(f"tempo de processamento: {tempo_total:.2f} segundos")

iniciando processamento...
concluido
total linhas: 40001454
tempo de processamento: 16.39 segundos


In [13]:
caminho_parquet = "dados/resultado_processado.parquet"
df_final.write.mode("overwrite").parquet()

## Desafio comparar o tempo entre parquet e csv

In [22]:
start_time = time.time()
caminho_parquet = "dados/resultado_processado.parquet"
df_parquet = spark.read.parquet(caminho_parquet)

print(f"Tempo para mapear o arquivo (Lazy): {time.time() - start_time:.6f} segundos")
print(f"Número estimado de partições: {df_parquet.rdd.getNumPartitions()}")


Tempo para mapear o arquivo (Lazy): 0.058002 segundos
Número estimado de partições: 22


In [23]:
df_parquet.printSchema()

root
 |-- id: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- status: string (nullable = true)
 |-- transacao_hash: string (nullable = true)
 |-- valor_float: float (nullable = true)



In [24]:
df_filtrado = df_parquet.filter(col("valor_float") > 200)

In [25]:
df_filtrado = df_filtrado.withColumn(
    "status_upper",
    upper(col("status"))
)

In [26]:
print("iniciando processamento...")
start_proc = time.time()

total_linhas = df_filtrado.count()

tempo_total = time.time() - start_proc
print(f"total linhas: {total_linhas}")
print(f"tempo de processamento: {tempo_total:.6f} segundos")

iniciando processamento...
total linhas: 31999881
tempo de processamento: 0.733894 segundos


In [27]:
df_filtrado.show(50)

+---+------------------+--------+--------------------+-----------+------------+
| id|             valor|  status|      transacao_hash|valor_float|status_upper|
+---+------------------+--------+--------------------+-----------+------------+
|  0| 555.8546829144523|Aprovado|e7b5eb1a-8014-45f...|   555.8547|    APROVADO|
|  6| 492.4743742878533|Aprovado|8c3c8dca-074b-484...|  492.47437|    APROVADO|
|  7|  644.274349771111|Aprovado|0f131ffb-af2a-43e...|  644.27435|    APROVADO|
|  8|  985.240453534253|Aprovado|ba48df0a-1843-413...|   985.2405|    APROVADO|
|  9| 749.4931997308543|Aprovado|fe32267b-3b89-4e2...|   749.4932|    APROVADO|
| 10| 566.1156380388716|Aprovado|c0f96983-5744-474...|  566.11566|    APROVADO|
| 14| 335.1184991122895|Aprovado|e8524976-3978-43c...|   335.1185|    APROVADO|
| 15| 935.3775940684937|Aprovado|920fce0a-6815-460...|   935.3776|    APROVADO|
| 17| 503.8857476605497|Aprovado|1f428089-2b69-453...|  503.88574|    APROVADO|
| 18|  652.716845888972|Aprovado|1fcfb63